In [1]:
pip install MFDFA

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
from MFDFA import MFDFA

# Base de datos: Consumo eléctrico

Lectura de base de datos

In [3]:
df = pd.read_csv('/content/drive/MyDrive/Tesis/Data/consumption.csv')

## 1) Variables endógenas o exógenas

Se imprime cada una de las variables del dataset

In [4]:
#Se imprime el nombre de las columnas
print(df.columns)

Index(['DateTime', 'Temperature', 'Humidity', 'Wind Speed',
       'general diffuse flows', 'diffuse flows', 'Zone 1 Power Consumption',
       'Zone 2  Power Consumption', 'Zone 3  Power Consumption'],
      dtype='object')


Sabiendo que las variables exógenas son aquellas que se originan fuera del sistema, por lo cual en este caso estas serían las siguientes 8:

* DateTime: Fecha
* Temperature: Temperatura
* Humidity: Humedad
* Wind Speed: Velocidad del viento
* general diffuse flows: Variable climatológica 1
* diffuse flows: punto de Variable climatológica 2
* Zone 2 Power Consumption: Consumo eléctrico en zona 2
* Zone 3 Power Consumption: Consumo eléctrico en zona 3



Por su parte, como se busca predecir una variable en particular, se cuenta con una única variable endógena, es decir:

* Zone 1 Power Consumption: Consumo eléctrico en zona 1




## 2) Problema Univariante o multivariante

Como se apreció previamente, el problema involucra a múltiples variables en estudio, por lo tanto se clasifica al problema como **multivariante**.

## 3) Muestreo Regular o Irregular

In [5]:
#Se copia el dataframe original
df2 = df.copy()

In [6]:
df2['DateTime'] = pd.to_datetime(df2['DateTime'], format='%m/%d/%Y %H:%M')

# Se establece la columna DateTime como index
df2.set_index('DateTime', inplace=False)

,Temperature,Humidity,Wind Speed,general diffuse flows,diffuse flows,Zone 1 Power Consumption,Zone 2 Power Consumption,Zone 3 Power Consumption
DateTime,,,,,,,,
2017-01-01 00:00:00,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386
2017-01-01 00:10:00,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434
2017-01-01 00:20:00,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373
2017-01-01 00:30:00,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711
2017-01-01 00:40:00,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964
...,...,...,...,...,...,...,...,...
2017-12-30 23:10:00,7.010,72.4,0.080,0.040,0.096,31160.45627,26857.31820,14780.31212
2017-12-30 23:20:00,6.947,72.6,0.082,0.051,0.093,30430.41825,26124.57809,14428.81152
2017-12-30 23:30:00,6.900,72.8,0.086,0.084,0.074,29590.87452,25277.69254,13806.48259


In [7]:
# Calcular la diferencia entre tiempos consecutivos
df2['date_diff'] = df2['DateTime'].diff()

# Revisar si todas las diferencias son iguales
regular = df2['date_diff'].iloc[1:].nunique() == 1

if regular:
    print("El muestreo es regular.")
else:
    print("El muestreo es irregular.")

El muestreo es regular.


## 4) Número de filas y columnas

Para encontrar el número de filas y columnas se utiliza el siguiente código.

In [8]:
#Se imprime la cantidad de filas y columnas
print(df.shape)

(52416, 9)


En total se cuenta con **52416 filas** y **9 columnas**.

## 5) Serie estacionaria o no estacionaria




Para considerar o no si la serie es estacionaria, se aplica la prueba ADF a cada serie de tiempo por separado que compone el dataframe, y luego se revisa si existe una mayor cantidad de series estacionarias o no estacionarias. Se utiliza un nivel de significación de 5%.

Se considera que las hipótesis son las siguientes:



*   H0: La serie tiene raíz unitaria, por lo que se considera no estacionaria.
*   H1: La serie no tiene raíz unitaria, por lo que se considera estacionaria.

Se salta la columna "DateTime"



In [9]:
# Función para aplicar el test ADF y mostrar los resultados
def adf_test(series):
    result = adfuller(series, autolag='AIC')
    print(f'ADF Statistic: {result[0]}')
    print(f'p-value: {result[1]}')


In [10]:
#Se copia el dataframe original
df3 = df.copy()
df3 = df3.drop(columns = ["DateTime"])

In [11]:
# Aplicar el test ADF a cada columna
for column in df3.columns:
    print(f'\nResultados del test ADF para la columna: {column}')
    adf_test(df3[column])


Resultados del test ADF para la columna: Temperature
ADF Statistic: -9.459827585705543
p-value: 4.384185727809841e-16

Resultados del test ADF para la columna: Humidity
ADF Statistic: -17.184247931293218
p-value: 6.616075836617442e-30

Resultados del test ADF para la columna: Wind Speed
ADF Statistic: -6.982259551899957
p-value: 8.135107819700119e-10

Resultados del test ADF para la columna: general diffuse flows
ADF Statistic: -34.86081721945389
p-value: 0.0

Resultados del test ADF para la columna: diffuse flows
ADF Statistic: -35.398974222363556
p-value: 0.0

Resultados del test ADF para la columna: Zone 1 Power Consumption
ADF Statistic: -32.12127853462604
p-value: 0.0

Resultados del test ADF para la columna: Zone 2  Power Consumption
ADF Statistic: -25.22216377100371
p-value: 0.0

Resultados del test ADF para la columna: Zone 3  Power Consumption
ADF Statistic: -16.36686797515675
p-value: 2.8351330869042474e-29


Luego, se llega a que todas las series se consideran estacionarias al rechazar la hipótesis nula, por lo que el dataframe se considera como estacionario.

## 6) Complejidad

Para definir la complejidad del conjunto de series de tiempo, de acuerdo a lo propuesto en la metodología original, se utiliza el método MF-DFA. Luego, como el problema es multivariante, se calcula la complejidad para cada serie de tiempo y finalmente se promedia el resultado.

Se define la cantidad de lags de acuerdo con lo expuesto en la documentación de la biblioteca MF-DFA utilizada. En otras palabras, el límite inferior es equivalente a el orden usado + 1 (en este caso es 3), con el límite superior siendo aproximadamente la cuarta parte de la cantidad total de muestras (12500 aproximadamente). Siguiendo los ejemplos, los lags se generan logarítimicamente.

In [ ]:
# Selección de lags
lag = np.unique(np.logspace(0.5, 4.1, 50).astype(int))

print(lag)

[    3     4     5     6     7     8    10    12    14    17    20    24
    28    33    39    47    56    66    78    93   110   130   154   183
   217   257   304   360   427   505   599   709   840   995  1178  1396
  1653  1958  2319  2746  3252  3852  4562  5403  6399  7578  8975 10629
 12589]


Siendo los exponentes fractales a utilizar, la lista de "q" se genera con valores entre 0 y 10 sin incluir al 0. Se podrían usar valores negativos, pero probando diferentes valores se llega a que lo más adecuado es utilizar valores positivos, de modo de evitar errores.

In [ ]:
# Selección de lista de poderes q
q_list = np.linspace(0,10,15)
q_list = q_list[q_list!=0.0]

print(q_list)

[ 0.71428571  1.42857143  2.14285714  2.85714286  3.57142857  4.28571429
  5.          5.71428571  6.42857143  7.14285714  7.85714286  8.57142857
  9.28571429 10.        ]


Se prueban diferentes órdenes del polinomio a aproximar, llegando a que el más adecuado es 2.

In [ ]:
# Orden de ajuste polinomial
order = 2

Se realiza el análisis para cada serie de tiempo.

In [15]:
# Diccionario para almacenar los resultados
resultados = {}
promedios = []
# Iterar sobre cada columna del DataFrame
for col in df3.columns:
    # Convertir la columna a un array de valores
    serie = df3[col].values

    # Aplicar el MF-DFA
    lag_values, dfa = MFDFA(serie, lag=lag, q=q_list, order=order)
    # Calcular los exponentes de Hurst (H)
    H_values = []
    for i, q in enumerate(q_list):
        H = np.polyfit(np.log(lag_values), np.log(dfa[:, i]), 1)[0]
        H_values.append(H)
    # Almacenar el valor promedio de H para la columna actual
    media_H = round(np.mean(H_values), 2)
    resultados[col] = media_H
    promedios.append(media_H)

# Imprimir los resultados
for col, H in resultados.items():
    print(f"El MFDFA para la columna {col} es: {H}")

# Calcular e imprimir el promedio de los promedios
promedio_global = round(np.mean(promedios), 2)
print(f"El promedio global de los promedios de MFDFA es: {promedio_global}")


El MFDFA para la columna Temperature es: 1.2
El MFDFA para la columna Humidity es: 1.2
El MFDFA para la columna Wind Speed es: 1.21
El MFDFA para la columna general diffuse flows es: 0.92
El MFDFA para la columna diffuse flows es: 0.85
El MFDFA para la columna Zone 1 Power Consumption es: 1.0
El MFDFA para la columna Zone 2  Power Consumption es: 1.07
El MFDFA para la columna Zone 3  Power Consumption es: 1.06
El promedio global de los promedios de MFDFA es: 1.06


En conclusión, el MFDFA alcanzado es de 1.06, que se puede interpretar como una baja complejidad.